In [ ]:
# !pip install recbole surprise kagglehub
# !pip install kmeans_pytorch
import numpy as np

_NUMPY_REMOVED_ALIASES = {
    'float_': np.float64,
    'complex_': np.complex128,
    'int_': np.int64,
    'longfloat': np.longdouble,
    'singlecomplex': np.complex64,
    'cfloat': np.complex128,
    'clongfloat': np.clongdouble,
    'string_': np.bytes_,
    'unicode_': np.str_,
    'object0': np.object_,
    'bytes0': np.bytes_,
    'str0': np.str_,
    'int0': np.intp,
    'uint0': np.uintp,
    'void0': np.void,
}
for _alias, _replacement in _NUMPY_REMOVED_ALIASES.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _replacement)

In [ ]:
import os
import ast
import json
import time
import logging
import pandas as pd
import torch
import kagglehub

from sklearn.model_selection import train_test_split

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import LightGCN
from recbole.model.context_aware_recommender.deepfm import DeepFM
from recbole.model.context_aware_recommender.widedeep import WideDeep
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger
from recbole.utils.case_study import full_sort_topk

from surprise import SVD, Dataset as SurpriseDataset, Reader

In [ ]:
mlpath = kagglehub.dataset_download("sherinclaudia/movielens")
print("Path to MovieLens files:", mlpath)

krpath = kagglehub.dataset_download("arashnic/kuairec-recommendation-system-data-density-100")
print("Path to KuaiRec files:", krpath)

Using Colab cache for faster access to the 'movielens' dataset.
Path to MovieLens files: /kaggle/input/movielens
Using Colab cache for faster access to the 'kuairec-recommendation-system-data-density-100' dataset.
Path to KuaiRec files: /kaggle/input/kuairec-recommendation-system-data-density-100


In [ ]:
kr_bigm = pd.read_csv("/kaggle/input/kuairec-recommendation-system-data-density-100/KuaiRec 2.0/data/big_matrix.csv")
kr_smallm = pd.read_csv("/kaggle/input/kuairec-recommendation-system-data-density-100/KuaiRec 2.0/data/small_matrix.csv")
kr_cats = pd.read_csv("/kaggle/input/kuairec-recommendation-system-data-density-100/KuaiRec 2.0/data/item_categories.csv")

ml_movies = pd.read_csv("/kaggle/input/movielens/movies.dat", sep='::',
                         names=['MovieID', 'Title', 'Genres'], engine='python', encoding='latin-1')
ml_movies['Genres'] = ml_movies['Genres'].fillna('').str.split('|').apply(lambda x: [g for g in x if g])

ml_ratings = pd.read_csv("/kaggle/input/movielens/ratings.dat", sep='::',
                          names=["UID", "MovieID", "Rating", "Timestamp"], engine='python', encoding='latin-1')

ml_users = pd.read_csv("/kaggle/input/movielens/users.dat", sep='::',
                        names=["UID", "SEX", "AGE", "OCC", "PIN"], engine='python', encoding='latin-1')

print(ml_movies.shape, ml_ratings.shape, ml_users.shape)

(3883, 3) (1000209, 4) (6040, 5)


In [ ]:
assert set(kr_smallm.user_id).issubset(set(kr_bigm.user_id))
assert set(kr_smallm.video_id).issubset(set(kr_bigm.video_id))
print("KuaiRec: small_matrix fully covered by big_matrix vocabulary OK")

KuaiRec: small_matrix fully covered by big_matrix vocabulary OK


In [ ]:
KR_THRESHOLD = 2.0
ML_THRESHOLD = 3.5

kr_bigm['label'] = (kr_bigm['watch_ratio'] > KR_THRESHOLD).astype(int)
kr_smallm['label'] = (kr_smallm['watch_ratio'] > KR_THRESHOLD).astype(int)
print("KuaiRec positive rates:", kr_bigm['label'].mean(), kr_smallm['label'].mean())

ml_ratings['label'] = (ml_ratings['Rating'] > ML_THRESHOLD).astype(int)
print("MovieLens positive rate:", ml_ratings['label'].mean())

KuaiRec positive rates: 0.07472703671256263 0.04643894991414648
MovieLens positive rate: 0.5751607913945985


In [ ]:
kr_user2idx = {u: i for i, u in enumerate(sorted(kr_bigm.user_id.unique()))}
kr_item2idx = {v: i for i, v in enumerate(sorted(kr_bigm.video_id.unique()))}

for df in (kr_bigm, kr_smallm):
    df['user_idx'] = df['user_id'].map(kr_user2idx)
    df['item_idx'] = df['video_id'].map(kr_item2idx)

n_kr_items = len(kr_item2idx)

ml_user2idx = {u: i for i, u in enumerate(sorted(ml_ratings.UID.unique()))}
ml_item2idx = {m: i for i, m in enumerate(sorted(ml_ratings.MovieID.unique()))}

ml_ratings['user_idx'] = ml_ratings['UID'].map(ml_user2idx)
ml_ratings['item_idx'] = ml_ratings['MovieID'].map(ml_item2idx)

n_ml_items = len(ml_item2idx)

In [ ]:
ml_train, ml_test = train_test_split(
    ml_ratings, test_size=0.2, random_state=42, stratify=ml_ratings['label']
)
print(f"MovieLens: {len(ml_train)} train / {len(ml_test)} test interactions")

MovieLens: 800167 train / 200042 test interactions


In [ ]:
kr_train_pos_pairs = set(
    zip(kr_bigm.loc[kr_bigm['label'] == 1, 'user_idx'],
        kr_bigm.loc[kr_bigm['label'] == 1, 'item_idx'])
)
kr_eval_pos_pairs = set(
    zip(kr_smallm.loc[kr_smallm['label'] == 1, 'user_idx'],
        kr_smallm.loc[kr_smallm['label'] == 1, 'item_idx'])
)
overlap_pairs = kr_train_pos_pairs & kr_eval_pos_pairs
print(f"KuaiRec: {len(overlap_pairs)} (user,item) positive pairs appear in BOTH "
      f"train and eval out of {len(kr_eval_pos_pairs)} eval positives "
      f"({100 * len(overlap_pairs) / max(len(kr_eval_pos_pairs), 1):.2f}%)")

kr_smallm['pair'] = list(zip(kr_smallm['user_idx'], kr_smallm['item_idx']))
kr_smallm_clean = kr_smallm[~kr_smallm['pair'].isin(overlap_pairs)].drop(columns=['pair']).reset_index(drop=True)
print(f"KuaiRec eval set: {len(kr_smallm)} rows -> {len(kr_smallm_clean)} rows after masking overlap")

KuaiRec: 0 (user,item) positive pairs appear in BOTH train and eval out of 217175 eval positives (0.00%)
KuaiRec eval set: 4676570 rows -> 4676570 rows after masking overlap


In [ ]:
def make_recbole_dirs(base_path, dataset_names):
    for name in dataset_names:
        os.makedirs(os.path.join(base_path, name), exist_ok=True)

def write_inter_file(df, path, user_col='user_idx', item_col='item_idx', label_col='label'):
    out = df[[user_col, item_col, label_col]].copy()
    out.columns = ['user_id:token', 'item_id:token', 'label:float']
    out.to_csv(path, sep='\t', index=False)

def write_item_file_multivalued(item_ids, feat_lists, path, feat_field='genre'):
    rows = []
    for iid, feats in zip(item_ids, feat_lists):
        feat_str = ' '.join(str(f) for f in feats) if len(feats) > 0 else ''
        rows.append((iid, feat_str))
    out = pd.DataFrame(rows, columns=['item_id:token', f'{feat_field}:token_seq'])
    out.to_csv(path, sep='\t', index=False)


BASE = '/content/recbole_data'
make_recbole_dirs(BASE, ['ml-1m', 'kuairec'])

ml_valid = ml_train.sample(frac=0.1, random_state=42)
ml_train_final = ml_train.drop(ml_valid.index)

write_inter_file(ml_train_final, f'{BASE}/ml-1m/ml-1m.train.inter')
write_inter_file(ml_valid,       f'{BASE}/ml-1m/ml-1m.valid.inter')
write_inter_file(ml_test,        f'{BASE}/ml-1m/ml-1m.test.inter')

write_item_file_multivalued(
    ml_movies['MovieID'].map(ml_item2idx).dropna().astype(int).tolist(),
    ml_movies.loc[ml_movies['MovieID'].map(ml_item2idx).notna(), 'Genres'].tolist(),
    f'{BASE}/ml-1m/ml-1m.item',
    feat_field='genre'
)
n_kept = ml_movies['MovieID'].map(ml_item2idx).notna().sum()
print(f"MovieLens items kept in .item file: {n_kept}/{len(ml_movies)}")

# ---- KuaiRec: train=big_matrix, eval=leakage-masked small_matrix ----
kr_valid = kr_bigm.sample(frac=0.02, random_state=42)
kr_train_final = kr_bigm.drop(kr_valid.index)

write_inter_file(kr_train_final, f'{BASE}/kuairec/kuairec.train.inter')
write_inter_file(kr_valid,       f'{BASE}/kuairec/kuairec.valid.inter')
write_inter_file(kr_smallm_clean, f'{BASE}/kuairec/kuairec.test.inter')   # leakage-masked

kr_cats['feat'] = kr_cats['feat'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
kr_items_mask = kr_cats['video_id'].map(kr_item2idx).notna()
write_item_file_multivalued(
    kr_cats.loc[kr_items_mask, 'video_id'].map(kr_item2idx).astype(int).tolist(),
    kr_cats.loc[kr_items_mask, 'feat'].tolist(),
    f'{BASE}/kuairec/kuairec.item',
    feat_field='feat'
)
n_kr_kept = kr_items_mask.sum()
print(f"KuaiRec items kept in .item file: {n_kr_kept}/{len(kr_cats)}")

print("Atomic files written.")
print(pd.read_csv(f'{BASE}/ml-1m/ml-1m.item', sep='\t').head())
print(pd.read_csv(f'{BASE}/kuairec/kuairec.item', sep='\t').head())

MovieLens items kept in .item file: 3706/3883
KuaiRec items kept in .item file: 10728/10728
Atomic files written.
   item_id:token               genre:token_seq
0              0   Animation Children's Comedy
1              1  Adventure Children's Fantasy
2              2                Comedy Romance
3              3                  Comedy Drama
4              4                        Comedy
   item_id:token feat:token_seq
0              0              8
1              1           27 9
2              2              9
3              3             26
4              4              5


In [ ]:
def compute_recall_ndcg_at_k(user_topk: dict, user_ground_truth: dict, k=20):
    recalls, ndcgs = [], []
    for user, ranked_items in user_topk.items():
        gt = user_ground_truth.get(user, set())
        if not gt:
            continue
        ranked_k = ranked_items[:k]
        hits = np.array([1 if item in gt else 0 for item in ranked_k])

        recall = hits.sum() / min(len(gt), k)
        recalls.append(recall)

        dcg = np.sum(hits / np.log2(np.arange(2, k + 2)))
        ideal_hits = min(len(gt), k)
        idcg = np.sum(1.0 / np.log2(np.arange(2, ideal_hits + 2))) if ideal_hits > 0 else 0.0
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return {f'Recall@{k}': np.mean(recalls), f'NDCG@{k}': np.mean(ndcgs)}

In [ ]:
RESULTS_PATH = '/content/results_log.csv'
RESULT_SCHEMA = ['dataset', 'model', 'seed', 'Recall@20', 'NDCG@20', 'error']

def load_completed_runs():
    """Only rows with NO error count as 'done' -- a failed row must not
    block a retry once the underlying bug is fixed."""
    if os.path.exists(RESULTS_PATH):
        df = pd.read_csv(RESULTS_PATH)
        successful = df[df['error'].isna()]
        return set(zip(successful['dataset'], successful['model'], successful['seed']))
    return set()

def append_result(row: dict):
    full_row = {col: row.get(col, np.nan) for col in RESULT_SCHEMA}
    df = pd.DataFrame([full_row], columns=RESULT_SCHEMA)
    header = not os.path.exists(RESULTS_PATH)
    df.to_csv(RESULTS_PATH, mode='a', header=header, index=False)

def clean_failed_rows():
    """Drop stale failure rows so successes always win and the CSV stays tidy."""
    if not os.path.exists(RESULTS_PATH):
        return
    df = pd.read_csv(RESULTS_PATH)
    for col in RESULT_SCHEMA:
        if col not in df.columns:
            df[col] = np.nan
    df = df[RESULT_SCHEMA]
    success = df[df['error'].isna()]
    failed = df[df['error'].notna()].drop_duplicates(subset=['dataset', 'model', 'seed'], keep='last')
    success_keys = set(zip(success['dataset'], success['model'], success['seed']))
    failed = failed[~failed.apply(lambda r: (r['dataset'], r['model'], r['seed']) in success_keys, axis=1)]
    out = pd.concat([success, failed], ignore_index=True)
    out.to_csv(RESULTS_PATH, index=False)
    print(f"Cleaned {RESULTS_PATH}: {len(success)} successes, {len(failed)} pending failures")

clean_failed_rows()

Cleaned /content/results_log.csv: 1 successes, 10 pending failures


In [ ]:
def build_recbole_config(dataset_name, model_name, use_content, emb_dim=64, seed=0):
    base_load_col = {'inter': ['user_id', 'item_id', 'label']}
    if use_content:
        feat_field = 'genre' if dataset_name == 'ml-1m' else 'feat'
        base_load_col['item'] = ['item_id', feat_field]

    config_dict = {
        'data_path': '/content/recbole_data',
        'dataset': dataset_name,
        'load_col': base_load_col,
        'embedding_size': emb_dim,
        'mlp_hidden_size': [128, 64],
        'train_neg_sample_args': {
            'distribution': 'uniform', 'sample_num': 1,
            'alpha': 1.0, 'dynamic': False, 'candidate_num': 0
        } if model_name == 'LightGCN' else None,
        'epochs': 15,
        'train_batch_size': 2048,
        'eval_batch_size': 4096,
        'learning_rate': 1e-3,
        'weight_decay': 1e-6,
        'seed': seed,
        'reproducibility': True,
        'eval_args': {
            'split': {'RS': None},
            'group_by': 'user',
            'order': 'RO',
            'mode': 'full',
        },
        'metrics': ['Recall', 'NDCG'],
        'topk': [20],
        'valid_metric': 'NDCG@20',
        'stopping_step': 3,
        'benchmark_filename': ['train', 'valid', 'test'],
        'field_separator': '\t',
        'USER_ID_FIELD': 'user_id',
        'ITEM_ID_FIELD': 'item_id',
        'LABEL_FIELD': 'label',
    }
    if use_content:
        feat_field = 'genre' if dataset_name == 'ml-1m' else 'feat'
        config_dict['selected_features'] = [feat_field]

    model_cls = {'DeepFM': DeepFM, 'WideDeep': WideDeep, 'LightGCN': LightGCN}[model_name]
    config = Config(model=model_cls, dataset=dataset_name, config_dict=config_dict)
    return config, model_cls


def train_recbole_model(dataset_name, model_name, use_content, seed, emb_dim=64):
    config, model_cls = build_recbole_config(dataset_name, model_name, use_content, emb_dim, seed)
    init_seed(config['seed'], config['reproducibility'])
    init_logger(config)
    logging.getLogger().setLevel(logging.WARNING)

    dataset = create_dataset(config)
    train_data, valid_data, test_data = data_preparation(config, dataset)

    model = model_cls(config, train_data.dataset).to(config['device'])
    trainer = Trainer(config, model)
    trainer.fit(train_data, valid_data, verbose=False, show_progress=False)

    return model, trainer, dataset, test_data, config


def get_recbole_topk_predictions(model, trainer, test_data, config, k=20):
    user_topk = {}
    uid_field = config['USER_ID_FIELD']
    all_users = test_data.dataset.inter_feat[uid_field].unique().cpu().numpy()

    model.eval()
    batch_size = 512
    for start in range(0, len(all_users), batch_size):
        batch_uids = all_users[start:start + batch_size]
        uid_series = torch.tensor(batch_uids, dtype=torch.long).to(config['device'])
        try:
            topk_scores, topk_iid_list = full_sort_topk(
                uid_series, model, test_data, k=k, device=config['device']
            )
        except RuntimeError as e:
            print(f"full_sort_topk failed on batch starting at {start}: {e}")
            raise
        topk_iid_list = topk_iid_list.cpu().numpy()
        for i, u in enumerate(batch_uids):
            user_topk[int(u)] = topk_iid_list[i].tolist()

    return user_topk


def get_ground_truth(dataset_name, split='test'):
    path = f'/content/recbole_data/{dataset_name}/{dataset_name}.{split}.inter'
    df = pd.read_csv(path, sep='\t')
    pos = df[df['label:float'] == 1.0]
    return pos.groupby('user_id:token')['item_id:token'].apply(set).to_dict()

In [ ]:
def train_surprise_svd(train_df, n_factors=64, seed=0):
    reader = Reader(rating_scale=(0, 1))
    data = SurpriseDataset.load_from_df(train_df[['user_idx', 'item_idx', 'label']], reader)
    trainset = data.build_full_trainset()
    algo = SVD(n_factors=n_factors, random_state=seed)
    algo.fit(trainset)
    return algo

def get_surprise_topk_predictions(algo, train_user_pos, all_item_ids, eval_users, k=20):
    user_topk = {}
    for u in eval_users:
        seen = train_user_pos.get(u, set())
        candidates = [i for i in all_item_ids if i not in seen]
        scores = [(i, algo.predict(u, i).est) for i in candidates]
        scores.sort(key=lambda x: -x[1])
        user_topk[u] = [i for i, _ in scores[:k]]
    return user_topk

In [ ]:
DATASETS = ['ml-1m', 'kuairec']
CONFIGS = [
    ('DeepFM', True), ('DeepFM', False),
    ('WideDeep', True), ('WideDeep', False),
    ('LightGCN', None),
    ('SVD', None),
]
SEEDS = [0, 1, 2, 3, 4]

def run_name(model_name, use_content):
    if use_content is None:
        return model_name
    return f"{model_name}_{'on' if use_content else 'off'}"

def run_all(datasets=DATASETS, configs=CONFIGS, seeds=SEEDS, k=20):
    completed = load_completed_runs()

    for dataset_name in datasets:
        train_path = f'/content/recbole_data/{dataset_name}/{dataset_name}.train.inter'
        train_df_raw = pd.read_csv(train_path, sep='\t')
        train_df_raw.columns = ['user_idx', 'item_idx', 'label']
        train_user_pos = train_df_raw[train_df_raw['label'] == 1.0].groupby('user_idx')['item_idx'].apply(set).to_dict()
        all_item_ids = sorted(train_df_raw['item_idx'].unique().tolist())
        ground_truth = get_ground_truth(dataset_name, split='test')
        eval_users = list(ground_truth.keys())

        for model_name, use_content in configs:
            name = run_name(model_name, use_content)

            for seed in seeds:
                if (dataset_name, name, seed) in completed:
                    print(f"SKIP (already done): {dataset_name} / {name} / seed={seed}")
                    continue

                print(f"\n=== {dataset_name} | {name} | seed={seed} ===")
                try:
                    t0 = time.time()
                    if model_name == 'SVD':
                        algo = train_surprise_svd(train_df_raw, n_factors=64, seed=seed)
                        user_topk = get_surprise_topk_predictions(
                            algo, train_user_pos, all_item_ids, eval_users, k=k
                        )
                    else:
                        model, trainer, ds, test_data, config = train_recbole_model(
                            dataset_name, model_name,
                            use_content=bool(use_content) if use_content is not None else False,
                            seed=seed
                        )
                        user_topk = get_recbole_topk_predictions(model, trainer, test_data, config, k=k)

                    metrics = compute_recall_ndcg_at_k(user_topk, ground_truth, k=k)
                    row = {'dataset': dataset_name, 'model': name, 'seed': seed, **metrics}
                    append_result(row)
                    print(row, f"| {time.time()-t0:.1f}s")

                except Exception as e:
                    print(f"FAILED: {dataset_name} / {name} / seed={seed} -- {e}")
                    append_result({'dataset': dataset_name, 'model': name, 'seed': seed,
                                    f'Recall@{k}': np.nan, f'NDCG@{k}': np.nan, 'error': str(e)})

    print("\nAll runs attempted. Results in", RESULTS_PATH)

In [ ]:
# run_all(seeds=[0])
# run_all(seeds=[1, 2, 3, 4])   # extend after seed 0 completes cleanly; resumable